# 05 · Optuna 파라미터·피처 개발

초기 100사이클 정보로 Cycle Life 회귀. 4개 모델 × 6개 피처 조합 × 원 수명/로그 수명 학습을 TPE로 탐색했다. 각 모델은 5개 외부 fold에서 35 trial, 최종 개발 데이터에서 80 trial로 총 1,020 trial이다. Test 점수로 모델·피처를 선정하거나 재보정하지 않는다. PDF 생성 없음.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
from IPython.display import display, Image
ROOT=Path.cwd()
if ROOT.name=="notebooks": ROOT=ROOT.parent
A=ROOT/"results/optuna"
summary=json.loads((A/"run_summary.json").read_text())
display(summary["selection"])
display(summary["feature_sets"])

## 탐색 범위

SVR: kernel(rbf/linear), C(로그 타깃 .01~100 / 원 타깃 1~10,000), gamma(.001~10), epsilon(로그 .001~.2 / 원 1~100). CatBoost: iterations 100~500, depth 1~5, learning_rate .005~.2, L2 .1~50, random_strength 0~3, RMSE/MAE 손실. Forests: trees 100/200, depth 1~10, min leaf 1~8, max features .4~1, criterion 제곱/절대 오차, bootstrap 여부.

피처·타깃 변환도 매 내부 CV에서 함께 선택한다. S4는 초기 IR 변화·Qd 기울기·충전 시간·정규화 ΔQ 면적을 추가했다. imputer/scaler는 fold 내부 fit. 4-fold 그룹 내부 CV 평균 원 단위 MAPE 최소화, 5-fold 그룹 외부 CV 평가. trial pruning은 사용하지 않아 모든 trial을 같은 4개 fold로 비교한다. Optuna study는 순차 실행하고 모델 내부 CPU는 2개 사용한다.

In [ ]:
display(pd.read_csv(A/"best_trials.csv").query("stage == 'final'"))
display(Image(filename=str(A/"optimization_history.png")))

**코멘트:** 최저 내부 CV 점수는 탐색에 사용했으므로 낙관적일 수 있다. 단순 피처와 복잡한 피처, 타깃 변환을 동일한 탐색 절차 안에서 비교한다. 이전 EDA 및 AutoML 결과를 보고 이 후보군을 정했으므로 새 nested CV도 전체 분석 과정의 선택 편향까지 제거하지는 못한다.

In [ ]:
display(pd.read_csv(A/"cv_comparison.csv"))
display(Image(filename=str(A/"nested_cv.png")))
display(pd.read_csv(A/"nested_cv_folds.csv"))

**코멘트:** 모델 계열별 행은 해당 계열 안에서 피처·파라미터를 선택한 절차의 외부 CV. SelectionProcedure는 모델 계열까지 내부 점수로 선택한 전체 절차의 외부 CV이다. fold 표준편차는 95% 신뢰구간이 아니다. 바깥 CV 순위를 보고 다시 모델을 선택하면 추가 선택 편향이 생긴다.

In [ ]:
display(pd.read_csv(A/"model_comparison.csv"))
display(pd.read_csv(A/"baseline_comparison.csv"))
display(Image(filename=str(A/"baseline_comparison.png")))
display(pd.read_csv(A/"performance_reporting.csv"))

**코멘트:** train은 재대입 점수이고 Train(CV)는 SelectionProcedure 외부 CV를 사용한다. Batch 2·3은 이미 이전 실험에서 보았으므로 완전 미열람 테스트가 아니다. 반복 탐색으로 낮아진 내부 점수와 외부 성능 개선을 구분한다. 테스트 결과에 맞춰 모델/예측을 교정하지 않는다. 원논문 9.1%는 데이터 전처리·라벨·분할이 같은지 미확인이라 참고 비교다.

In [ ]:
display(json.loads((A/"promotion_decision.json").read_text()))

**유지 결정:** 이번 후보는 내부 CV 7.907%로 이전 AutoML 7.537%보다 나쁘고, 전체 선택 절차 외부 CV도 12.653%로 이전 12.211%보다 나쁘다. 개발 데이터 기준에서도 개선되지 않아 기본 추천 모델은 기존 results/automl/final_model.joblib를 유지한다. Batch 2 악화는 별도 평가 관측이며 교체 결정 규칙에는 사용하지 않는다. 이는 사전 등록 검정이 아니라 실험 후의 보수적 유지 판단이다. 넓은 탐색 공간에서 제한된 trial은 이전 좋은 설정을 놓칠 수 있고, 피처·타깃·파라미터 선택이 작은 개발 표본에 과적합될 수 있다. 원인을 확정하는 결과는 아니다.

In [ ]:
display(pd.read_csv(A/"error_analysis.csv").head(10))
import joblib
from src.preprocess import load_data
df=load_data(ROOT/"data/battery_modeling_ready.parquet")
model=joblib.load(A/"final_model.joblib")
p=model.predict(df)
assert np.isfinite(p).all()
display(pd.DataFrame({"battery_id":df.battery_id,"predicted_cycle_life":p}).head())

## Colab에서 재실행

ZIP을 /content/ess-battery-project로 풀고 프로젝트 루트에서 아래 선택 셀을 실행한다. 저장된 노트북 결과를 보는 데는 재탐색이 필요 없다. 기존 study를 덮어쓰지 않도록 다른 --out을 지정한다. joblib은 신뢰할 수 있는 파일만 로드한다. 수명 label은 제공 cycle_life이고 실제 EOL 기준은 미확인이다.

In [ ]:
# %cd /content/ess-battery-project
# %pip install -r requirements-optuna.txt
# !python -m src.tune_optuna --out results/optuna_colab --outer-trials 35 --final-trials 80